# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/manalchaudharyy/FlyrankAI-ML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.***1. What one row means for my lane:** for my feature/label frame, one row = one content
item (`content_hash_id`), aggregated over a single calendar month. The raw fact table's
native grain is one row per content item per client per day — I roll that up to
content-per-month for Lane 2 scoring.

**2. Table(s):** `fact_content_daily_performance` (daily metrics, source of my features and
label) joined to `dim_content` (content metadata: word count, age, content type) on
`content_hash_id`.

**3. Time window:** `month=2026-03` (a mid-panel month, per this week's warning — never the
`_sample`/final month for developing label logic). Feature window = first half of March
(2026-03-01 to 2026-03-15); label proxy compares to the second half (2026-03-16 to 2026-03-31),
so my feature window never overlaps my label window.

**4. Target/proxy:** `is_declining_proxy` = 1 if a content item's total clicks in the second
half of March are lower than in the first half, else 0. This is a within-month directional
proxy, not a validated future-decline label — I'm reusing the same "current-window bucket"
caution the starter dataset's `trend_direction` label carries (from w01/w02).

**5. One thing I deliberately exclude:** `fact_content_query_90d` (the query-level table).
I'm keeping this pass to page-level daily facts only — adding query-mix features this early
adds join complexity and its own leakage risks (rare/anonymized query tails) I haven't
audited yet. I'll revisit it once the baseline (page-level) contract is solid.

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
FACT_MONTH = f"{BASE}/fact_content_daily_performance/month=2026-03/*.parquet"
DIM_CONTENT = f"{BASE}/dim_content/*.parquet"

# First, always inspect the schema before writing queries -- don't guess column names
print(con.sql(f"DESCRIBE SELECT * FROM read_parquet('{FACT_MONTH}') LIMIT 1"))

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

In [23]:
print(con.sql(f"DESCRIBE SELECT * FROM read_parquet('{FACT_MONTH}') LIMIT 1"))

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

In [24]:
schema_df = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{FACT_MONTH}') LIMIT 1").df()
import pandas as pd
pd.set_option('display.max_rows', None)
print(schema_df[['column_name', 'column_type']].to_string())

                 column_name column_type
0                report_date        DATE
1             client_hash_id     VARCHAR
2            content_hash_id     VARCHAR
3             client_has_gsc     BOOLEAN
4             client_has_ga4     BOOLEAN
5         gsc_data_available     BOOLEAN
6         ga4_data_available     BOOLEAN
7            gsc_impressions      BIGINT
8                 gsc_clicks      BIGINT
9           gsc_sum_position      BIGINT
10          gsc_avg_position      DOUBLE
11             ga4_pageviews      BIGINT
12              ga4_sessions      BIGINT
13                 ga4_users      BIGINT
14      ga4_engaged_sessions      BIGINT
15  ga4_total_engagement_sec      BIGINT
16          sessions_organic      BIGINT
17           sessions_direct      BIGINT
18         sessions_referral      BIGINT
19           sessions_social      BIGINT
20             sessions_paid      BIGINT
21               sessions_ai      BIGINT
22                ai_chatgpt      BIGINT
23             a

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.***Feature:** `impressions`, `clicks`, `avg_position` (computed from position*impressions/impressions),
`sessions` (GA4), `word_count` and `content_age_days` (from dim_content) — all measured
inside the FIRST half of March only.

**Label:** `is_declining_proxy` — computed by comparing first-half vs second-half clicks
(see Section 1). Calculated only from observed outcomes, not a product decision flag.

**Context:** `content_hash_id`, `client_hash_id`, `content_type`/intent (for grouping and
later fairness checks) — used for joins and grouping, never as a raw model feature.

**Excluded:** `sessions_ai` — per the lane guide's density warning (30,177 rows with AI
sessions vs 78.8M total), too sparse to use as a feature or label component this week.
Also excluded: any rebuilt product flag (`health_score`, `priority_score`) — not shipped in
this data, so nothing to accidentally leak in.

In [25]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [26]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Prove the grain: does (content_hash_id, report_date) uniquely identify a row?
q1 = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(DISTINCT content_hash_id || '|' || report_date) AS unique_grain_keys
    FROM read_parquet('{FACT_MONTH}')
""").df()
print(q1)
print("If total_rows == unique_grain_keys, the grain claim (one row = one content item per day) holds.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  unique_grain_keys
0     9841378            9841378
If total_rows == unique_grain_keys, the grain claim (one row = one content item per day) holds.


In [27]:
q2 = con.sql(f"""
    SELECT COUNT(*) AS row_count,
           MIN(report_date) AS earliest_date,
           MAX(report_date) AS latest_date,
           COUNT(DISTINCT content_hash_id) AS unique_content_items
    FROM read_parquet('{FACT_MONTH}')
""").df()
print(q2)

   row_count earliest_date latest_date  unique_content_items
0    9841378    2026-03-01  2026-03-31                331437


In [28]:
q3 = con.sql(f"""
    SELECT
        COUNT(*) AS all_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS rows_with_ga4,
        ROUND(100.0 * COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) / COUNT(*), 1) AS pct_with_ga4
    FROM read_parquet('{FACT_MONTH}')
""").df()
print(q3)
print("This tells me what fraction of March rows actually have GA4 sessions available --")
print("important before I treat a missing session as a real zero.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   all_rows  rows_with_ga4  pct_with_ga4
0   9841378         413966           4.2
This tells me what fraction of March rows actually have GA4 sessions available --
important before I treat a missing session as a real zero.


In [29]:
features_df = con.sql(f"""
    SELECT
        f.content_hash_id,
        SUM(f.gsc_impressions) AS impressions_h1,
        SUM(f.gsc_clicks) AS clicks_h1,
        SUM(f.gsc_clicks) / NULLIF(SUM(f.gsc_impressions), 0) AS ctr_h1,
        AVG(f.gsc_avg_position) AS avg_position_h1,
        SUM(f.ga4_sessions) FILTER (WHERE f.ga4_data_available IS TRUE) AS sessions_h1
    FROM read_parquet('{FACT_MONTH}') f
    WHERE f.report_date BETWEEN '2026-03-01' AND '2026-03-15'
    GROUP BY f.content_hash_id
""").df()

print(features_df.shape)
features_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(319759, 6)


,content_hash_id,impressions_h1,clicks_h1,ctr_h1,avg_position_h1,sessions_h1
0,content_d0dff76c889de68f,111.0,0.0,0.000000,5.222776,NaN
1,content_67741cce996cfafa,38.0,1.0,0.026316,4.638889,NaN
2,content_2e6360ad20fd7107,219.0,1.0,0.004566,3.737399,NaN
3,content_ac8663da7484669a,20.0,0.0,0.000000,3.597222,NaN
4,content_65c50dfe9d87a585,1494.0,0.0,0.000000,6.156643,NaN


In [30]:
print("""
Feature availability notes:
1. impressions_h1      -> gsc_impressions, known as soon as GSC reports the first half of March.
2. clicks_h1            -> gsc_clicks, observed within the feature window only.
3. ctr_h1                -> derived purely from the two features above, same window.
4. avg_position_h1      -> gsc_avg_position, observed within the feature window.
5. sessions_h1           -> ga4_sessions, observed within the feature window, filtered to
                            rows where GA4 tracking was actually live (IS TRUE check).
""")


Feature availability notes:
1. impressions_h1      -> gsc_impressions, known as soon as GSC reports the first half of March.
2. clicks_h1            -> gsc_clicks, observed within the feature window only.
3. ctr_h1                -> derived purely from the two features above, same window.
4. avg_position_h1      -> gsc_avg_position, observed within the feature window.
5. sessions_h1           -> ga4_sessions, observed within the feature window, filtered to
                            rows where GA4 tracking was actually live (IS TRUE check).



In [31]:
label_df = con.sql(f"""
    SELECT
        content_hash_id,
        SUM(gsc_clicks) AS clicks_h2
    FROM read_parquet('{FACT_MONTH}')
    WHERE report_date BETWEEN '2026-03-16' AND '2026-03-31'
    GROUP BY content_hash_id
""").df()

data = features_df.merge(label_df, on="content_hash_id", how="inner")
data["is_declining_proxy"] = (data["clicks_h2"] < data["clicks_h1"]).astype(int)

print(data.shape)
print("Declining rate:", data["is_declining_proxy"].mean().round(3))
data.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(319758, 8)
Declining rate: 0.091


,content_hash_id,impressions_h1,clicks_h1,ctr_h1,avg_position_h1,sessions_h1,clicks_h2,is_declining_proxy
0,content_d0dff76c889de68f,111.0,0.0,0.000000,5.222776,NaN,0.0,0
1,content_67741cce996cfafa,38.0,1.0,0.026316,4.638889,NaN,0.0,1
2,content_2e6360ad20fd7107,219.0,1.0,0.004566,3.737399,NaN,0.0,1
3,content_ac8663da7484669a,20.0,0.0,0.000000,3.597222,NaN,0.0,0
4,content_65c50dfe9d87a585,1494.0,0.0,0.000000,6.156643,NaN,0.0,0


In [32]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

feature_cols = ["impressions_h1", "clicks_h1", "ctr_h1", "avg_position_h1", "sessions_h1"]
X = data[feature_cols].fillna(0)
y = data["is_declining_proxy"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
honest_model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
honest_auc = roc_auc_score(y_test, honest_model.predict_proba(X_test)[:, 1])
print(f"Honest AUC (5 features only): {honest_auc:.3f}")

# --- THE TRAP: sneak in a column derived straight from the label ---
data["leaky_clicks_h2"] = data["clicks_h2"]
X_leak = data[feature_cols + ["leaky_clicks_h2"]].fillna(0)
X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(X_leak, y, test_size=0.3, random_state=42)
leaky_model = LogisticRegression(max_iter=1000).fit(X_train_l, y_train_l)
leaky_auc = roc_auc_score(y_test_l, leaky_model.predict_proba(X_test_l)[:, 1])
print(f"LEAKY AUC (with clicks_h2 sneaked in): {leaky_auc:.3f}  <- jumps toward 1.0, looks amazing, means nothing")

# --- Delete the leak, keep the honest number ---
del data["leaky_clicks_h2"]
print(f"\nFinal honest number I'm keeping: AUC = {honest_auc:.3f}")

Honest AUC (5 features only): 0.945
LEAKY AUC (with clicks_h2 sneaked in): 1.000  <- jumps toward 1.0, looks amazing, means nothing

Final honest number I'm keeping: AUC = 0.945


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*This March 2026 slice can never tell me: how this pattern behaves in other seasons (only one
month observed, and the lane guide notes clients have unbalanced history depth — some as
short as a few months); whether a "decline" here is real decline versus consolidation to a
sibling page (I didn't check related content_hash_id groups this week); or anything about
clients whose GA4 tracking hadn't started yet in March (Query 3 shows the % of rows without
GA4 -- those content items only have GSC-side features, which biases sessions_h1 toward
clients with earlier GA4 rollout). I also can't claim this proxy label predicts a REAL
future outcome, since it splits one month in half rather than using a genuine
past-window -> future-window design.

In [33]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.